In [1]:
# ==============================
# Federated Learning on EMNIST Digits (Dirichlet Non-IID)
# Model: LeNet++ CNN
# Algorithm: FedProx + Masking + Adaptive Noise
# Tracks: Train/Val/Test Loss & Accuracy + Round Time + Total Time
# ==============================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import random
import copy
from collections import OrderedDict
import time

# -------------------------
# Configuration
# -------------------------
NUM_CLIENTS = 100
CLIENT_FRAC = 0.3
ROUNDS = 60
LOCAL_EPOCHS = 3
BATCH_SIZE = 128
LR = 0.01
MU = 0.01
MASK_SCALE = 0.001      # cryptographic masking
ADAPTIVE_NOISE_FACTOR = 0.1  # set 0 to verify masking doesn't affect accuracy
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
ALPHA = 0.1  # very strict non-IID

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.benchmark = True

print("Device:", DEVICE)

# -------------------------
# Dataset: EMNIST Digits
# -------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.EMNIST(
    root="./data", split="digits", train=True, download=True, transform=transform
)
test_dataset = datasets.EMNIST(
    root="./data", split="digits", train=False, download=True, transform=transform
)

# -------------------------
# Validation Split: 10 fixed clients
# -------------------------
VAL_CLIENTS = list(range(10))
TRAIN_CLIENTS = list(range(10, NUM_CLIENTS))

# -------------------------
# Dirichlet Non-IID Partition
# -------------------------
def iid_partition(dataset, clients):
    idxs = np.arange(len(dataset))
    np.random.shuffle(idxs)

    split = np.array_split(idxs, len(clients))

    client_dict = {i: list(split[i]) for i in clients}
    return client_dict

client_data = iid_partition(train_dataset, list(range(NUM_CLIENTS)))

# -------------------------
# Model: CNN (LeNet++)
# -------------------------
class EMNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64*7*7, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# -------------------------
# FedProx Regularizer
# -------------------------
def fedprox_loss(model, global_model, mu):
    prox = 0.0
    for w, w0 in zip(model.parameters(), global_model.parameters()):
        prox += torch.sum((w - w0.detach()) ** 2)
    return mu/2 * prox

# -------------------------
# Model <-> Vector Utils (for masking + noise)
# -------------------------
def model_to_vector(model):
    return torch.cat([p.data.view(-1) for p in model.parameters()])

def vector_to_model(model, vec):
    pointer = 0
    for p in model.parameters():
        num = p.numel()
        p.data = vec[pointer:pointer+num].view_as(p).clone()
        pointer += num

# -------------------------
# Evaluation
# -------------------------
def evaluate(model, loader):
    model.eval()
    correct, total, loss_val = 0, 0, 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss_val += F.cross_entropy(out, y).item() * y.size(0)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100 * correct / total, loss_val / total

# -------------------------
# Federated Training
# -------------------------
global_model = EMNIST_CNN().to(DEVICE)
global_vec = model_to_vector(global_model)

train_acc, val_acc, test_acc = [], [], []
loss_curve, train_loss_curve, val_loss_curve, test_loss_curve = [], [], [], []
round_time, total_time = [], []

start_total = time.time()

for rnd in range(ROUNDS):
    start_round = time.time()
    selected_clients = random.sample(TRAIN_CLIENTS, int(NUM_CLIENTS*CLIENT_FRAC))

    # ------------------------
    # Ring-based pairwise masks
    # ------------------------
    masks = {cid: torch.zeros_like(global_vec) for cid in selected_clients}
    sorted_ids = sorted(selected_clients)
    K = len(sorted_ids)
    for idx, cid in enumerate(sorted_ids):
        left = sorted_ids[(idx - 1) % K]
        right = sorted_ids[(idx + 1) % K]
        if cid < right:
            rmask = torch.randn_like(global_vec) * MASK_SCALE
            masks[cid] += rmask
            masks[right] -= rmask

    updates, local_losses = [], []

    for cid in selected_clients:
        local_model = copy.deepcopy(global_model)
        local_model.train()

        optimizer = optim.SGD(local_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
        loader = DataLoader(Subset(train_dataset, client_data[cid]), batch_size=BATCH_SIZE, shuffle=True)

        for _ in range(LOCAL_EPOCHS):
            for x, y in loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()
                loss = F.cross_entropy(local_model(x), y)
                loss += fedprox_loss(local_model, global_model, MU)
                loss.backward()
                optimizer.step()

        local_losses.append(loss.item())

        # ------------------------
        # Mask + Adaptive Noise
        # ------------------------
        delta = model_to_vector(local_model) - global_vec
        pointer = 0
        for p in local_model.parameters():
            num = p.numel()
            layer_update = delta[pointer:pointer+num].view_as(p)
            layer_norm = torch.norm(layer_update)
            noise = torch.randn_like(layer_update) * layer_norm * ADAPTIVE_NOISE_FACTOR
            delta[pointer:pointer+num] += noise.view(-1)
            pointer += num

        # Add mask
        delta += masks[cid]
        updates.append(delta)

    # Aggregate updates
    avg_update = torch.stack(updates).mean(0)
    global_vec += avg_update
    vector_to_model(global_model, global_vec)

    # ------------------------
    # Metrics
    # ------------------------
    tr_acc, tr_loss = evaluate(global_model, DataLoader(
        torch.utils.data.ConcatDataset([Subset(train_dataset, client_data[c]) for c in selected_clients]), batch_size=BATCH_SIZE))
    va_acc, va_loss = evaluate(global_model, DataLoader(
        torch.utils.data.ConcatDataset([Subset(train_dataset, client_data[c]) for c in VAL_CLIENTS]), batch_size=BATCH_SIZE))
    te_acc, te_loss = evaluate(global_model, DataLoader(test_dataset, batch_size=BATCH_SIZE))

    train_acc.append(tr_acc)
    val_acc.append(va_acc)
    test_acc.append(te_acc)
    loss_curve.append(np.mean(local_losses))
    train_loss_curve.append(tr_loss)
    val_loss_curve.append(va_loss)
    test_loss_curve.append(te_loss)

    rt = time.time() - start_round
    round_time.append(rt)
    total_time.append(time.time() - start_total)

    print(f"Round {rnd+1:03d} | "
          f"LocalLoss {loss_curve[-1]:.4f} | "
          f"TrainLoss {tr_loss:.4f} | "
          f"ValLoss {va_loss:.4f} | "
          f"TestLoss {te_loss:.4f} | "
          f"Train {tr_acc:.4f} | Val {va_acc:.4f} | Test {te_acc:.4f} | "
          f"RoundTime {rt:.2f}s | TotalTime {total_time[-1]:.2f}s")

Device: cuda


100%|██████████| 562M/562M [00:08<00:00, 63.3MB/s]


Round 001 | LocalLoss 0.4530 | TrainLoss 0.9773 | ValLoss 0.9789 | TestLoss 0.9727 | Train 65.0042 | Val 65.3208 | Test 65.2900 | RoundTime 85.50s | TotalTime 85.50s
Round 002 | LocalLoss 0.2075 | TrainLoss 0.2485 | ValLoss 0.2555 | TestLoss 0.2490 | Train 92.5569 | Val 92.2208 | Test 92.3825 | RoundTime 80.66s | TotalTime 166.16s
Round 003 | LocalLoss 0.1438 | TrainLoss 0.1520 | ValLoss 0.1618 | TestLoss 0.1545 | Train 95.4653 | Val 95.0917 | Test 95.3450 | RoundTime 81.45s | TotalTime 247.61s
Round 004 | LocalLoss 0.1165 | TrainLoss 0.1320 | ValLoss 0.1419 | TestLoss 0.1350 | Train 96.0625 | Val 95.7708 | Test 95.7875 | RoundTime 80.91s | TotalTime 328.52s
Round 005 | LocalLoss 0.0902 | TrainLoss 0.1077 | ValLoss 0.1201 | TestLoss 0.1124 | Train 96.7764 | Val 96.3667 | Test 96.4925 | RoundTime 81.49s | TotalTime 410.01s
Round 006 | LocalLoss 0.0717 | TrainLoss 0.0942 | ValLoss 0.1043 | TestLoss 0.0975 | Train 97.1722 | Val 96.9458 | Test 97.0075 | RoundTime 80.51s | TotalTime 490.52s

In [2]:
# -------------------------
# Save metrics to CSV
# -------------------------
import csv

metrics = {
    "round": list(range(1, ROUNDS+1)),
    "train_acc": train_acc,
    "val_acc": val_acc,
    "test_acc": test_acc,
    "local_loss": loss_curve,
    "train_loss": train_loss_curve,
    "val_loss": val_loss_curve,
    "test_loss": test_loss_curve,
    "round_time": round_time,
    "total_time": total_time
}

with open("IID_EMNIST_Digits_masking_federated_metrics_1.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(metrics.keys())
    writer.writerows(zip(*metrics.values()))

# Colab download
try:
    from google.colab import files
    files.download("IID_EMNIST_Digits_masking_federated_metrics_1.csv")
except:
    print("CSV saved locally as IID_EMNIST_Digits_masking_federated_metrics_1.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>